### Ingesting Product Master

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    DateType,
    IntegerType
)

In [0]:
# Defining Schema

product_master_schema = StructType([
    StructField("product_id",      StringType(),  nullable=True),   
    StructField("product_name",    StringType(),  nullable=True),  
    StructField("category",        StringType(),  nullable=True),   
    StructField("subcategory",     StringType(),  nullable=True),   
    StructField("brand",           StringType(),  nullable=True),   
    StructField("supplier_id",     StringType(),  nullable=True),   
    StructField("cost_price",      DoubleType(),  nullable=True),
    StructField("selling_price",   DoubleType(),  nullable=True),   
    StructField("weight",          DoubleType(),  nullable=True),   
    StructField("dimensions",      StringType(),  nullable=True),   
    StructField("status",          StringType(),  nullable=True),   
    StructField("effective_date",  StringType(),  nullable=True),    
    StructField("expiry_date",     IntegerType(),  nullable=True),   
])

In [0]:
from pyspark.sql import functions as F

# Reading from raw
df_stream = (
    spark.readStream
        .format("parquet")
        .schema(product_master_schema)
        .load("/Volumes/mini_project_team_a3t7/default/mini_project/raw/product_master/")
)

# Adding Metadata columns
df_bronze = (
    df_stream
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_batch_date", F.current_date())
        .withColumn("_source", F.lit("product_master_parquet"))
        
        # Add Watermark
        .withWatermark("_ingestion_timestamp", "10 minutes")
)

# Write Stream
query = (
    df_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/bronze/product_master/"
        )
        .partitionBy("_batch_date")
        .trigger(availableNow=True)
        .start("/Volumes/mini_project_team_a3t7/default/mini_project/bronze/product_master/")
)

### Ingesting Store Master

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType,
)

In [0]:
# Defining Schema
store_master_schema = StructType([
    StructField("store_id",     StringType(), nullable=False),
    StructField("store_name",   StringType(), nullable=True),
    StructField("region",       StringType(), nullable=True),
    StructField("city",         StringType(), nullable=True),
    StructField("store_type",   StringType(), nullable=True),
    StructField("opening_date", DateType(),   nullable=True),
])

In [0]:
from pyspark.sql import functions as F

# Reading the data
raw_df = (
    spark.readStream
        .format("cloudFiles")               # Use Auto Loader
        .option("cloudFiles.format", "csv") # Specify input format
        .schema(store_master_schema)
        .option("header", "true")
        .option("nullValue", "")
        .load("/Volumes/mini_project_team_a3t7/default/mini_project/raw/store_master/store_master.csv")
)

# Adding Metadata column + Watermark
enriched_df = (
    raw_df
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_batch_date", F.current_date())
        .withColumn("_source", F.lit("pos_batch_csv"))
        
        # Watermark added here
        .withWatermark("_ingestion_timestamp", "10 minutes")
)

# Writing in bronze table
write_query = (
    enriched_df.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/bronze/store_master/"
        )
        .partitionBy("_batch_date")
        .trigger(availableNow=True)
        .start("/Volumes/mini_project_team_a3t7/default/mini_project/bronze/Store_Master/")
)

### Ingesting Supply Master

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType,
)

In [0]:
# Defining Schema
supplier_master_schema = StructType([
    StructField("supplier_id",           StringType(),  nullable=False),
    StructField("supplier_name",         StringType(),  nullable=True),
    StructField("category",              StringType(),  nullable=True),
    StructField("performance_rating",    DoubleType(),  nullable=True),
    StructField("on_time_delivery_pct",  DoubleType(),  nullable=True),
])

In [0]:
from pyspark.sql import functions as F

# Reading the data
raw_df = (
    spark.readStream
        .format("cloudFiles")               # Use Auto Loader
        .option("cloudFiles.format", "csv") # Specify input format
        .schema(supplier_master_schema)
        .option("header", "true")
        .option("nullValue", "")
        .load("/Volumes/mini_project_team_a3t7/default/mini_project/raw/supplier_master/supplier_master.csv")
)

# Adding metadata columns + watermark
enriched_df = (
    raw_df
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_batch_date", F.current_date())
        .withColumn("_source", F.lit("pos_batch_csv"))
        
        # Watermark added here
        .withWatermark("_ingestion_timestamp", "10 minutes")
)

# Writing to bronze table
write_query = (
    enriched_df.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/bronze/Supply_Master/"
        )
        .partitionBy("_batch_date")
        .trigger(availableNow=True)
        .start("/Volumes/mini_project_team_a3t7/default/mini_project/bronze/supply_master/")
)